# M1. 데이터 확인 (01_data_check)

`data/raw`의 원본 6개 파일을 읽어 **구조 · 결측 · 중복 · 기준연도 · 핵심 컬럼**을 확인합니다.
- 원본은 수정하지 않고 읽기만 합니다.
- 위에서부터 순서대로 실행하면 됩니다. 결과 요약은 `data/analysis/01_data_check_summary.csv`로 저장됩니다.

## 1. 준비: 프로젝트 경로와 파일 목록

In [ ]:
from pathlib import Path
import re
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# 노트북 위치와 상관없이 data/raw가 있는 폴더를 위로 올라가며 찾는다
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
assert RAW.exists(), "data/raw 폴더를 찾지 못했습니다. 프로젝트 폴더에서 노트북을 여세요."

print("프로젝트 폴더:", ROOT)
for p in sorted(RAW.iterdir()):
    if p.is_file() and not p.name.startswith("."):
        print(f"  {p.name}  ({p.stat().st_size/1024:,.0f} KB)")

## 2. 6개 파일 읽기
- 1번(xlsx): `개요` 시트는 비어 있고, 표는 두 시트에 있으며 머리글 위치가 다릅니다. 위치를 지정해서 읽습니다.
- 2~5번(CSV): cp949, 6번(CSV): utf-8-sig. 두 인코딩을 차례로 시도합니다.

In [ ]:
def find(num):
    """파일명이 '번호_'로 시작하는 CSV 1개를 찾는다."""
    hits = [p for p in RAW.iterdir() if p.is_file() and p.suffix.lower() == ".csv"
            and p.name.startswith(f"{num}_")]
    assert len(hits) == 1, f"{num}번 파일이 {len(hits)}개입니다: {[h.name for h in hits]}"
    return hits[0]

def csv_auto(path):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩을 읽지 못했습니다: {path.name}")

xlsx = [p for p in RAW.iterdir() if p.suffix.lower() in (".xlsx", ".xls") and not p.name.startswith(("~", "."))]
assert len(xlsx) == 1, f"엑셀 파일이 {len(xlsx)}개입니다: {[x.name for x in xlsx]}"
x1 = xlsx[0]

# 1번: 제조업등 시트(머리글 3번째 줄), 건설업 시트(머리글 줄이 따로 없어 열 이름 직접 지정)
mf = pd.read_excel(x1, sheet_name="아카이브(제조업등)", header=2, usecols="B:J")
mf.columns = [str(c).replace("\n", "").replace(" ", "") for c in mf.columns]
cs = pd.read_excel(x1, sheet_name="아카이브(건설업)", header=None, skiprows=4, usecols="B:J")
cs.columns = ["연번", "공종", "작업명", "단위작업명", "재해종류", "재해개요", "기인물", "재해유발요인", "위험성감소대책(예시)"]

dfs = {
    "1a_SIF_제조업등": mf,
    "1b_SIF_건설업": cs,
    "2_업종별규모별_재해자수": csv_auto(find(2)),
    "3_규모별_사망자수": csv_auto(find(3)),
    "4_업종별규모별_사업장수": csv_auto(find(4)),
    "5_업종별규모별_사망만인율": csv_auto(find(5)),
    "6_출처목록": csv_auto(find(6)),
}
for k, d in dfs.items():
    print(f"{k:<26} {d.shape[0]:>6,} 행 × {d.shape[1]:>2} 열")

## 3. 파일별 구조 확인 (shape · columns · head)

In [ ]:
for name, d in dfs.items():
    print("=" * 90)
    print(f"[{name}]  {d.shape[0]:,}행 × {d.shape[1]}열")
    print("컬럼:", list(d.columns))
    display(d.head(3))

## 4. 컬럼 타입과 비어 있지 않은 값 수 (info)

In [ ]:
for name, d in dfs.items():
    print("=" * 90)
    print(f"[{name}]")
    d.info()

## 5. 결측 · 중복 요약표

In [ ]:
rows = []
for name, d in dfs.items():
    n_cell = d.shape[0] * d.shape[1]
    n_na = int(d.isna().sum().sum())
    rows.append({
        "파일": name,
        "행": d.shape[0],
        "열": d.shape[1],
        "결측셀": n_na,
        "결측비율(%)": round(n_na / n_cell * 100, 2) if n_cell else 0,
        "결측있는컬럼수": int((d.isna().sum() > 0).sum()),
        "완전중복행": int(d.duplicated().sum()),
    })
summary = pd.DataFrame(rows)
display(summary)

In [ ]:
# 결측이 있는 컬럼만 파일별로 표시
for name, d in dfs.items():
    na = d.isna().sum()
    na = na[na > 0]
    print(f"[{name}]", "결측 없음" if na.empty else "")
    if not na.empty:
        display(pd.DataFrame({"결측수": na, "비율(%)": (na / len(d) * 100).round(1)}))

In [ ]:
# 1번 사례 데이터: 같은 재해개요가 반복되는지 확인 (삭제하지 않고 확인만)
for name in ["1a_SIF_제조업등", "1b_SIF_건설업"]:
    d = dfs[name]
    dup = d[d.duplicated("재해개요", keep=False) & d["재해개요"].notna()]
    print(f"[{name}] 재해개요가 같은 행: {len(dup)}개")
    if len(dup):
        display(dup.sort_values("재해개요")[["재해개요"]].head(10))

## 6. 기준 연도 확인
- 1번 사례: `재해개요` 앞부분에 적힌 연도(예: `2023년 ...`)를 추출합니다.
- 3번 통계: 컬럼 이름에 들어 있는 연도 범위를 확인합니다.

In [ ]:
YEAR = re.compile(r"(?<!\d)((?:19|20)\d{2})")

def case_years(s):
    y = s.dropna().astype(str).str.strip().str.extract(YEAR)[0].dropna().astype(int)
    return y

year_rows = []
for name in ["1a_SIF_제조업등", "1b_SIF_건설업"]:
    y = case_years(dfs[name]["재해개요"])
    year_rows.append({"대상": name + " (재해개요)", "연도 추출 건수": len(y),
                      "전체 대비(%)": round(len(y) / len(dfs[name]) * 100, 1),
                      "최소": y.min(), "최대": y.max()})
    print(f"[{name}] 연도별 건수")
    print(y.value_counts().sort_index().to_string(), "\n")

cols3 = [int(m.group(1)) for c in dfs["3_규모별_사망자수"].columns
         if (m := re.fullmatch(r"\s*((?:19|20)\d{2})\s*(?:년)?\s*", str(c)))]
if cols3:
    year_rows.append({"대상": "3번 통계 (컬럼)", "연도 추출 건수": len(cols3),
                      "전체 대비(%)": None, "최소": min(cols3), "최대": max(cols3)})
display(pd.DataFrame(year_rows))

## 7. 핵심 컬럼 · 키 후보 확인
RAG 문서가 되는 1번 사례 데이터에서 각 컬럼이 얼마나 채워져 있고 값이 몇 종류인지 봅니다.
통계 파일(2~5번)은 `구분`(업종)과 규모 컬럼으로 서로 조인합니다.

In [ ]:
for name in ["1a_SIF_제조업등", "1b_SIF_건설업"]:
    d = dfs[name]
    prof = pd.DataFrame({
        "채움(%)": (d.notna().mean() * 100).round(1),
        "고유값수": d.nunique(),
        "예시": d.apply(lambda s: str(s.dropna().iloc[0])[:40] if s.notna().any() else ""),
    })
    print(f"[{name}]")
    display(prof)

# 2~5번: 업종(구분) 컬럼과 값 확인
for name in ["2_업종별규모별_재해자수", "4_업종별규모별_사업장수", "5_업종별규모별_사망만인율"]:
    d = dfs[name]
    key = d.columns[0]
    print(f"[{name}] 첫 컬럼 '{key}' 값 {d[key].nunique()}종, 중복 {int(d[key].duplicated().sum())}개")
    print("  ", list(d[key].astype(str).head(8)), "...")

## 8. 데이터 명세 초안 (저장)

In [ ]:
spec = pd.DataFrame([
    ["1a_SIF_제조업등", "산재 사고사례(제조업 등)", "RAG 문서", "연번", "재해개요에서 연도 추출 필요"],
    ["1b_SIF_건설업",   "산재 사고사례(건설업)",     "RAG 문서", "연번", "재해개요에서 연도 추출 필요"],
    ["2_업종별규모별_재해자수",   "업종 × 규모 통계", "통계/차트", "구분(업종)", "업종명 표기 통일 필요"],
    ["3_규모별_사망자수",         "규모 × 연도 통계", "통계/차트", "규모",       "연도가 컬럼으로 펼쳐진 형태"],
    ["4_업종별규모별_사업장수",   "업종 × 규모 통계", "통계/차트", "구분(업종)", "사망만인율 해석에 사용"],
    ["5_업종별규모별_사망만인율", "업종 × 규모 통계", "통계/차트", "구분(업종)", "사업장수 0인 칸의 결측은 의미 있는 결측"],
    ["6_출처목록",     "데이터 출처 주소 목록", "참고(분석 제외)", "-", "사례 데이터가 아니라 출처 목록"],
], columns=["파일", "내용", "용도", "키 컬럼", "비고"])
display(spec)

OUT = ROOT / "data" / "analysis"
OUT.mkdir(parents=True, exist_ok=True)
result = summary.merge(spec, on="파일", how="left")
result.to_csv(OUT / "01_data_check_summary.csv", index=False, encoding="utf-8-sig")
print("저장:", (OUT / "01_data_check_summary.csv").relative_to(ROOT))

## 9. 결과 메모 (직접 채우기)
실행 결과를 보고 아래를 확인·수정해 두면 M2 전처리 기준이 됩니다.
- 결측이 많은 컬럼과 처리 방침 (삭제하지 않고 NaN 유지가 기본)
- 중복 행 / 같은 재해개요 사례 처리 방침
- 기준 연도 범위와 연도 추출이 안 되는 사례 수
- 1번(RAG 문서)과 2~5번(통계)의 역할 구분